Date: 15/10/2025
Desc: To find D_max based on newest complete data

## For MCS Adaptation

Finding D_max based on first h_dist at which no MCS meet reliability requirement.

In [10]:
import pandas as pd
import numpy as np

MCS_BITRATE_MAP = {0: 6.5, 1: 13, 2: 19.5, 3: 26, 4: 39, 5: 52, 6: 58.5, 7: 65} 

dl_df_gt = pd.read_csv("/media/research-student/DataDrive/FANET_Dataset/complete_testing_dmax_dataset/data_processed_complete/Downlink_Reliability.csv",)
ul_df_gt = pd.read_csv("/media/research-student/DataDrive/FANET_Dataset/complete_testing_dmax_dataset/data_processed_complete/Uplink_Reliability.csv")
vid_df_gt = pd.read_csv("/media/research-student/DataDrive/FANET_Dataset/complete_testing_dmax_dataset/data_processed_complete/Video_Reliability.csv")
dl_df_gt["Reliability"] = dl_df_gt["Num_Reliable"] / (dl_df_gt["Num_Delay_Excd"] + dl_df_gt["Num_Fail_Other"] + dl_df_gt["Num_Reliable"])
ul_df_gt["Reliability"] = ul_df_gt["Num_Reliable"] / (ul_df_gt["Num_Delay_Excd"] + ul_df_gt["Num_Fail_Other"] + ul_df_gt["Num_Reliable"])
vid_df_gt["Reliability"] = vid_df_gt["Num_Reliable"] / (vid_df_gt["Num_Delay_Excd"] + vid_df_gt["Num_Fail_Other"] + vid_df_gt["Num_Reliable"])

reliability_th = 0.9
reliability_th_str = "90" # Remember to change this when changing reliability_th
hdist_step_size = 5

heights = [75, 105, 135, 165, 195, 225, 255, 285]
usi_list = [10, 20, 66.7, 100]

results_list = []
for height in heights:
    for usi in usi_list:
        curr_hdist = 0
        stop = 0
        while not stop:
            # Get the df at curr_hdist for the specified height and mcs
            dl_gt_df_curr = dl_df_gt[(dl_df_gt["Height"] == height) & (dl_df_gt["UAV_Sending_Interval"] == usi) & (dl_df_gt["Horizontal_Distance"] == curr_hdist)]
            ul_gt_df_curr = ul_df_gt[(ul_df_gt["Height"] == height) & (ul_df_gt["UAV_Sending_Interval"] == usi) & (ul_df_gt["Horizontal_Distance"] == curr_hdist)]
            vid_gt_df_curr = vid_df_gt[(vid_df_gt["Height"] == height) & (vid_df_gt["UAV_Sending_Interval"] == usi) & (vid_df_gt["Horizontal_Distance"] == curr_hdist)]
            # Merge the reliabilities into dl_gt_df_curr
            dl_gt_df_curr = dl_gt_df_curr.sort_values(by=["UAV_Sending_Interval", "Height", "Horizontal_Distance"])
            ul_gt_df_curr = ul_gt_df_curr.sort_values(by=["UAV_Sending_Interval", "Height", "Horizontal_Distance"])
            vid_gt_df_curr = vid_gt_df_curr.sort_values(by=["UAV_Sending_Interval", "Height", "Horizontal_Distance"])
            dl_gt_df_curr = dl_gt_df_curr.rename(columns={"Reliability": "Reliability_DL"})
            dl_gt_df_curr["Reliability_UL"] = ul_gt_df_curr["Reliability"].values
            dl_gt_df_curr["Reliability_VID"] = vid_gt_df_curr["Reliability"].values
            dl_gt_df_curr["Reliability_State"] = (dl_gt_df_curr["Reliability_DL"] >= reliability_th) & (dl_gt_df_curr["Reliability_UL"] >= reliability_th) & (dl_gt_df_curr["Reliability_VID"] >= reliability_th)
            # Check if any reliability in df is below threshold
            if np.any(dl_gt_df_curr["Reliability_State"]):
                curr_hdist += hdist_step_size
            else:
                stop = 1
                if curr_hdist - hdist_step_size < 0:
                    results_list.append({"Height": height, "UAV_Sending_Interval": usi, "D_max": 0})
                else:
                    results_list.append({"Height": height, "UAV_Sending_Interval": usi, "D_max": curr_hdist - hdist_step_size})

results_df = pd.DataFrame(results_list)
results_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/mdp_max_hdist_Oct25/MDP_MCS_Max_HDist_fm_Sim_{}.csv".format(reliability_th_str), index=False)

## For USI Adaptation

Finding D_max based on first h_dist at which no USI meet reliability requirement.

In [4]:
import pandas as pd
import numpy as np

MCS_BITRATE_MAP = {0: 6.5, 1: 13, 2: 19.5, 3: 26, 4: 39, 5: 52, 6: 58.5, 7: 65} 

dl_df_gt = pd.read_csv("/media/research-student/DataDrive/FANET_Dataset/complete_testing_dmax_dataset/data_processed_complete/Downlink_Reliability.csv",)
ul_df_gt = pd.read_csv("/media/research-student/DataDrive/FANET_Dataset/complete_testing_dmax_dataset/data_processed_complete/Uplink_Reliability.csv")
vid_df_gt = pd.read_csv("/media/research-student/DataDrive/FANET_Dataset/complete_testing_dmax_dataset/data_processed_complete/Video_Reliability.csv")
dl_df_gt["Reliability"] = dl_df_gt["Num_Reliable"] / (dl_df_gt["Num_Delay_Excd"] + dl_df_gt["Num_Fail_Other"] + dl_df_gt["Num_Reliable"])
ul_df_gt["Reliability"] = ul_df_gt["Num_Reliable"] / (ul_df_gt["Num_Delay_Excd"] + ul_df_gt["Num_Fail_Other"] + ul_df_gt["Num_Reliable"])
vid_df_gt["Reliability"] = vid_df_gt["Num_Reliable"] / (vid_df_gt["Num_Delay_Excd"] + vid_df_gt["Num_Fail_Other"] + vid_df_gt["Num_Reliable"])

reliability_th = 0.90
reliability_th_str = "90" # Remember to change this when changing reliability_th
hdist_step_size = 5

heights = [75, 105, 135, 165, 195, 225, 255, 285]
mcs_list = np.arange(0, 8, 1)

results_list = []
for height in heights:
    for mcs in mcs_list:
        curr_hdist = 0
        stop = 0
        while not stop:
            # Get the df at curr_hdist for the specified height and mcs
            dl_gt_df_curr = dl_df_gt[(dl_df_gt["Height"] == height) & (dl_df_gt["Bitrate"] == MCS_BITRATE_MAP[mcs]) & (dl_df_gt["Horizontal_Distance"] == curr_hdist)]
            ul_gt_df_curr = ul_df_gt[(ul_df_gt["Height"] == height) & (ul_df_gt["Bitrate"] == MCS_BITRATE_MAP[mcs]) & (ul_df_gt["Horizontal_Distance"] == curr_hdist)]
            vid_gt_df_curr = vid_df_gt[(vid_df_gt["Height"] == height) & (vid_df_gt["Bitrate"] == MCS_BITRATE_MAP[mcs]) & (vid_df_gt["Horizontal_Distance"] == curr_hdist)]
            # Merge the reliabilities into dl_gt_df_curr
            dl_gt_df_curr = dl_gt_df_curr.sort_values(by=["Bitrate", "Height", "Horizontal_Distance"])
            ul_gt_df_curr = ul_gt_df_curr.sort_values(by=["Bitrate", "Height", "Horizontal_Distance"])
            vid_gt_df_curr = vid_gt_df_curr.sort_values(by=["Bitrate", "Height", "Horizontal_Distance"])
            dl_gt_df_curr = dl_gt_df_curr.rename(columns={"Reliability": "Reliability_DL"})
            dl_gt_df_curr["Reliability_UL"] = ul_gt_df_curr["Reliability"].values
            dl_gt_df_curr["Reliability_VID"] = vid_gt_df_curr["Reliability"].values
            dl_gt_df_curr["Reliability_State"] = (dl_gt_df_curr["Reliability_DL"] >= reliability_th) & (dl_gt_df_curr["Reliability_UL"] >= reliability_th) & (dl_gt_df_curr["Reliability_VID"] >= reliability_th)
            # Check if any reliability in df is below threshold
            if np.any(dl_gt_df_curr["Reliability_State"]):
                curr_hdist += hdist_step_size
            else:
                stop = 1
                if curr_hdist - hdist_step_size < 0:
                    results_list.append({"Height": height, "MCS": mcs, "D_max": 0})
                else:
                    results_list.append({"Height": height, "MCS": mcs, "D_max": curr_hdist - hdist_step_size})

results_df = pd.DataFrame(results_list)
results_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/mdp_max_hdist_Oct25/MDP_USI_Max_HDist_fm_Sim_{}.csv".format(reliability_th_str), index=False)

## For Both MCS and USI Adaptation

Finding D_max based on first h_dist at which no USI and MCS pair meet reliability requirement.

In [3]:
import pandas as pd
import numpy as np

MCS_BITRATE_MAP = {0: 6.5, 1: 13, 2: 19.5, 3: 26, 4: 39, 5: 52, 6: 58.5, 7: 65} 

dl_df_gt = pd.read_csv("/media/research-student/DataDrive/FANET_Dataset/complete_testing_dmax_dataset/data_processed_complete/Downlink_Reliability.csv",)
ul_df_gt = pd.read_csv("/media/research-student/DataDrive/FANET_Dataset/complete_testing_dmax_dataset/data_processed_complete/Uplink_Reliability.csv")
vid_df_gt = pd.read_csv("/media/research-student/DataDrive/FANET_Dataset/complete_testing_dmax_dataset/data_processed_complete/Video_Reliability.csv")
dl_df_gt["Reliability"] = dl_df_gt["Num_Reliable"] / (dl_df_gt["Num_Delay_Excd"] + dl_df_gt["Num_Fail_Other"] + dl_df_gt["Num_Reliable"])
ul_df_gt["Reliability"] = ul_df_gt["Num_Reliable"] / (ul_df_gt["Num_Delay_Excd"] + ul_df_gt["Num_Fail_Other"] + ul_df_gt["Num_Reliable"])
vid_df_gt["Reliability"] = vid_df_gt["Num_Reliable"] / (vid_df_gt["Num_Delay_Excd"] + vid_df_gt["Num_Fail_Other"] + vid_df_gt["Num_Reliable"])

reliability_th = 0.999
reliability_th_str = "999" # Remember to change this when changing reliability_th
hdist_step_size = 5

heights = [75, 105, 135, 165, 195, 225, 255, 285]

results_list = []
for height in heights:
    curr_hdist = 0
    stop = 0
    while not stop:
        # Get the df at curr_hdist for the specified height and mcs
        dl_gt_df_curr = dl_df_gt[(dl_df_gt["Height"] == height) & (dl_df_gt["Horizontal_Distance"] == curr_hdist)]
        ul_gt_df_curr = ul_df_gt[(ul_df_gt["Height"] == height) & (ul_df_gt["Horizontal_Distance"] == curr_hdist)]
        vid_gt_df_curr = vid_df_gt[(vid_df_gt["Height"] == height) & (vid_df_gt["Horizontal_Distance"] == curr_hdist)]
        # Merge the reliabilities into dl_gt_df_curr
        dl_gt_df_curr = dl_gt_df_curr.sort_values(by=["UAV_Sending_Interval", "Bitrate", "Height", "Horizontal_Distance"])
        ul_gt_df_curr = ul_gt_df_curr.sort_values(by=["UAV_Sending_Interval", "Bitrate", "Height", "Horizontal_Distance"])
        vid_gt_df_curr = vid_gt_df_curr.sort_values(by=["UAV_Sending_Interval", "Bitrate", "Height", "Horizontal_Distance"])
        dl_gt_df_curr = dl_gt_df_curr.rename(columns={"Reliability": "Reliability_DL"})
        dl_gt_df_curr["Reliability_UL"] = ul_gt_df_curr["Reliability"].values
        dl_gt_df_curr["Reliability_VID"] = vid_gt_df_curr["Reliability"].values
        dl_gt_df_curr["Reliability_State"] = (dl_gt_df_curr["Reliability_DL"] >= reliability_th) & (dl_gt_df_curr["Reliability_UL"] >= reliability_th) & (dl_gt_df_curr["Reliability_VID"] >= reliability_th)
        # Check if any reliability in df is below threshold
        if np.any(dl_gt_df_curr["Reliability_State"]):
            curr_hdist += hdist_step_size
        else:
            stop = 1
            if curr_hdist - hdist_step_size < 0:
                results_list.append({"Height": height, "D_max": 0})
            else:
                results_list.append({"Height": height, "D_max": curr_hdist - hdist_step_size})

results_df = pd.DataFrame(results_list)
results_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/mdp_max_hdist_Oct25/MDP_USInMCS_Max_HDist_fm_Sim_{}.csv".format(reliability_th_str), index=False)